# 09 — Upload the features, held-out logits and code to Zenodo

Run on a **CPU runtime**. It needs about **25 GB of free local disk**; building and uploading take roughly one to
three hours, depending on Colab's upload speed (an estimate).

What it does:
1. **Packages** the frozen features in `TMLR_v3/features/` (one zip per dataset) and the held-out logits in
   `results/pools/`, plus a snapshot of the GitHub repository.
2. **Leaves out the six gated pathology encoders** (UNI, UNI2-h, Virchow2, H-optimus-0, Prov-GigaPath, Hibou-B),
   both their features and their logits, as a precaution: their licences may restrict redistributing outputs. The
   list is read from `config.py` (`gated=True`).
3. **Verifies** every archive by re-hashing every file inside it, then **copies the archives to Drive**
   (`TMLR_v3/zenodo_export/`, checked by MD5), so a runtime reset never forces a rebuild.
4. **Creates a Zenodo draft, reserves its DOI and uploads everything.** Licence: **CC BY-NC 4.0**. The draft's id
   and DOI are saved to `TMLR_v3/zenodo_draft.json` the moment the draft exists. Each file is retried up to three
   times (Zenodo sometimes answers *504 Gateway Time-out*), and several files can upload at once.
   **It never publishes.** You check the draft on the website and press *Publish* yourself, because publishing is
   irreversible.

Your Zenodo token is read from Colab **Secrets** (key icon in the left sidebar; enable notebook access). The
notebook finds it under any of the usual names, and works out whether it is a zenodo.org or a sandbox token.

**After a disconnect:** run cells 1–2 (top), then Settings, the token, the build cell (it finds the copy on Drive
and skips the build) and the upload cell. The same draft and DOI are reused and finished files are skipped.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, sys, importlib, shutil, json
PROJECT = "/content/drive/MyDrive/TMLR_v3"
CODE = os.path.join(PROJECT, "code")
os.makedirs(CODE, exist_ok=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
print("project:", PROJECT)

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/export_zenodo.py
"""Package the frozen features and held-out logits for Zenodo, verify them, and
(optionally) upload them to a Zenodo *draft* that you review and publish yourself.

Layout produced (one archive per dataset keeps every file well under Zenodo's
limits and lets users download only what they need):

    features_<dataset>.zip   <dataset>/<encoder>.npy + <dataset>/<encoder>.json
    heldout_logits.zip       pools/<dataset>_logits.npz   (per-fold held-out logits)
    zenodo_manifest.json     SHA-256 of every archive and of every file inside it
    README.md                what the record contains and how to use it

The archives are written with ZIP_STORED: float32 arrays barely compress, and
stored members can be read back without decompression cost. Nothing is ever
modified in place; the source files on Drive are only read.
"""

from __future__ import annotations

import glob
import hashlib
import json
import os
import time
import zipfile

import config as CFG

CHUNK = 1 << 22


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for b in iter(lambda: fh.read(CHUNK), b""):
            h.update(b)
    return h.hexdigest()


def md5_file(path):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for b in iter(lambda: fh.read(CHUNK), b""):
            h.update(b)
    return h.hexdigest()


# ---------------------------------------------------------------- inventory
def inventory(exclude_datasets=(), exclude_encoders=()):
    """Every feature file (with its provenance json) and every logits file."""
    feats = {}
    for npy in sorted(glob.glob(os.path.join(CFG.FEAT_DIR, "*", "*.npy"))):
        ds = os.path.basename(os.path.dirname(npy))
        enc = os.path.splitext(os.path.basename(npy))[0]
        if ds in exclude_datasets or enc in exclude_encoders or enc.startswith("_"):
            continue
        meta = npy[:-4] + ".json"
        feats.setdefault(ds, []).append({"encoder": enc, "npy": npy,
                                         "json": meta if os.path.exists(meta) else None,
                                         "bytes": os.path.getsize(npy)})
    logits = [p for p in sorted(glob.glob(os.path.join(CFG.RESULTS_DIR, "pools", "*_logits.npz")))
              if os.path.basename(p)[:-len("_logits.npz")] not in exclude_datasets]
    return feats, logits


def summary(feats, logits):
    n = sum(len(v) for v in feats.values())
    b = sum(x["bytes"] for v in feats.values() for x in v) + sum(os.path.getsize(p) for p in logits)
    return {"datasets": len(feats), "feature_files": n, "logits_files": len(logits),
            "total_gb": round(b / 1e9, 2),
            "per_dataset": {ds: {"encoders": len(v), "gb": round(sum(x["bytes"] for x in v) / 1e9, 3)}
                            for ds, v in sorted(feats.items())}}


# ---------------------------------------------------------------- build
def build(out_dir, feats, logits, include_logits=True, exclude_logits_of=(), verbose=True):
    """Write one archive per dataset (+ logits) and the manifest. Resumable: an
    archive that already exists and verifies against its sidecar is skipped."""
    os.makedirs(out_dir, exist_ok=True)
    manifest = {"created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "archives": {},
                "encoders_excluded": sorted(exclude_logits_of)}
    jobs = [(f"features_{ds}.zip", [(x["npy"], f"{ds}/{x['encoder']}.npy") for x in items]
             + [(x["json"], f"{ds}/{x['encoder']}.json") for x in items if x["json"]])
            for ds, items in sorted(feats.items())]
    if include_logits and logits:
        if exclude_logits_of:
            import numpy as np
            tmp = os.path.join(out_dir, "_filtered_logits")
            os.makedirs(tmp, exist_ok=True)
            kept = []
            for p in logits:
                z = np.load(p)
                keep = {k: z[k] for k in z.files if not (k.startswith("Z_") and k[2:] in exclude_logits_of)}
                q = os.path.join(tmp, os.path.basename(p))
                np.savez_compressed(q, **keep)
                kept.append(q)
            logits = kept
        jobs.append(("heldout_logits.zip", [(p, f"pools/{os.path.basename(p)}") for p in logits]))
    for name, members in jobs:
        path = os.path.join(out_dir, name)
        side = path + ".members.json"
        if os.path.exists(path) and os.path.exists(side):
            rec = json.load(open(side))
            if rec.get("archive_sha256") == sha256_file(path):
                manifest["archives"][name] = rec
                if verbose:
                    print(f"  {name}: exists, verified - skipped", flush=True)
                continue
        t0 = time.time()
        inner = {}
        with zipfile.ZipFile(path + ".part", "w", zipfile.ZIP_STORED, allowZip64=True) as z:
            for src, arc in members:
                inner[arc] = {"sha256": sha256_file(src), "bytes": os.path.getsize(src)}
                z.write(src, arc)
        os.replace(path + ".part", path)
        rec = {"archive_sha256": sha256_file(path), "archive_md5": md5_file(path),
               "archive_bytes": os.path.getsize(path), "members": inner}
        with open(side, "w") as fh:
            json.dump(rec, fh, indent=1)
        manifest["archives"][name] = rec
        if verbose:
            print(f"  {name}: {len(inner)} files, {rec['archive_bytes'] / 1e9:.2f} GB "
                  f"({(time.time() - t0) / 60:.1f} min)", flush=True)
    with open(os.path.join(out_dir, "zenodo_manifest.json"), "w") as fh:
        json.dump(manifest, fh, indent=1)
    return manifest


def add_file(out_dir, path, name=None):
    """Copy a file into the export folder unchanged and record its checksums."""
    import shutil
    dst = os.path.join(out_dir, name or os.path.basename(path))
    shutil.copy2(path, dst)
    mp = os.path.join(out_dir, "zenodo_manifest.json")
    man = json.load(open(mp))
    man.setdefault("other_files", {})[os.path.basename(dst)] = {
        "sha256": sha256_file(dst), "md5": md5_file(dst), "bytes": os.path.getsize(dst)}
    with open(mp, "w") as fh:
        json.dump(man, fh, indent=1)
    return dst


def verify(out_dir, verbose=True):
    """Re-open every archive and re-hash every member against the manifest."""
    man = json.load(open(os.path.join(out_dir, "zenodo_manifest.json")))
    bad = []
    for name, rec in man["archives"].items():
        path = os.path.join(out_dir, name)
        if sha256_file(path) != rec["archive_sha256"]:
            bad.append((name, "archive hash"))
            continue
        with zipfile.ZipFile(path) as z:
            names = set(z.namelist())
            for arc, info in rec["members"].items():
                if arc not in names:
                    bad.append((name, f"missing {arc}"))
                    continue
                h = hashlib.sha256()
                with z.open(arc) as fh:
                    for b in iter(lambda: fh.read(CHUNK), b""):
                        h.update(b)
                if h.hexdigest() != info["sha256"]:
                    bad.append((name, f"hash {arc}"))
        if verbose:
            print(f"  {name}: {'OK' if not any(b[0] == name for b in bad) else 'FAILED'}", flush=True)
    return bad


def write_readme(out_dir, title, github_url, paper_title, summary_dict, excluded=(), code_snapshot=None):
    lines = [f"# {title}", "",
             f"Frozen-encoder features and per-fold held-out logits for the study *{paper_title}*.",
             "", f"Code, registration, per-pool records and the verification gate: {github_url}", "",
             "## Contents", "",
             f"- `features_<dataset>.zip`: one archive per dataset ({summary_dict['datasets']} datasets, "
             f"{summary_dict['feature_files']} feature files). Each holds `<dataset>/<encoder>.npy` (float32, "
             "rows in the order of the dataset manifest in the GitHub repository) and `<dataset>/<encoder>.json` "
             "(encoder source, preprocessing, pooling, load report, feature statistics, extraction time).",
             "- `heldout_logits.zip`: `pools/<dataset>_logits.npz`, the per-fold held-out probe logits of every "
             "encoder, with labels and fold masks, as used by the analysis.",
             "- `zenodo_manifest.json`: SHA-256 of every archive and of every file inside it.", ""]
    if code_snapshot:
        lines[-1:-1] = [f"- `{code_snapshot}`: snapshot of the code repository at the time of deposit."]
    if excluded:
        lines += ["## Encoders not included", "",
                  "As a precaution, features and held-out logits of the following encoders are not redistributed: "
                  "they are distributed under gated licences that may restrict redistributing their outputs: " + ", ".join(f"`{e}`" for e in excluded)
                  + ". They are regenerated from the released code after obtaining each model from its authors "
                  "(notebook 01).", ""]
    lines += [
             "## Use", "",
             "`python code/fetch_features.py --record <this record's id> --root <dir>` in the GitHub repository "
             "downloads, verifies and unpacks everything into the layout the notebooks expect.", "",
             "## Provenance and licences", "",
             "Features were extracted from publicly available datasets with publicly released encoders, "
             "some of them gated, under their respective licences. Users must comply with the terms of the "
             "original datasets and models; see the GitHub repository for sources.", "",
             "## Per dataset", "", "| dataset | encoders | GB |", "|---|---|---|"]
    lines += [f"| {ds} | {v['encoders']} | {v['gb']} |" for ds, v in summary_dict["per_dataset"].items()]
    with open(os.path.join(out_dir, "README.md"), "w") as fh:
        fh.write("\n".join(lines) + "\n")


# ---------------------------------------------------------------- Zenodo
def detect_site(token):
    """Return 'zenodo' or 'sandbox' for the site that accepts this token, or None."""
    import requests
    for site, base in (("zenodo", "https://zenodo.org/api"), ("sandbox", "https://sandbox.zenodo.org/api")):
        r = requests.get(f"{base}/deposit/depositions", params={"size": 1},
                         headers={"Authorization": f"Bearer {token}"}, timeout=60)
        if r.status_code == 200:
            return site
    return None


class Zenodo:
    """Minimal client for the Zenodo deposit REST API. Creates and fills a DRAFT;
    it never publishes (publishing on Zenodo is irreversible)."""

    def __init__(self, token, sandbox=False, session=None):
        import requests
        self.base = "https://sandbox.zenodo.org/api" if sandbox else "https://zenodo.org/api"
        self.web = self.base.replace("/api", "")
        self.s = session or requests.Session()
        self.s.headers["Authorization"] = f"Bearer {token}"

    def create_draft(self):
        r = self.s.post(f"{self.base}/deposit/depositions", json={"metadata": {"prereserve_doi": True}})
        r.raise_for_status()
        return r.json()

    def get(self, dep_id):
        r = self.s.get(f"{self.base}/deposit/depositions/{dep_id}")
        r.raise_for_status()
        return r.json()

    def uploaded(self, dep):
        return {f["filename"]: f.get("checksum") for f in dep.get("files", [])}

    def upload(self, dep, path, verbose=True, attempts=3, wait=20, local_md5=None):
        """Stream one file into the draft's bucket; Zenodo returns its MD5, which is
        compared with the local file's. A network error, a 5xx answer (e.g. a 504
        gateway time-out) or a checksum mismatch is retried up to `attempts` times;
        re-sending a file to the same name replaces it."""
        name = os.path.basename(path)
        local = local_md5 or md5_file(path)
        last = None
        for attempt in range(1, attempts + 1):
            t0 = time.time()
            try:
                with open(path, "rb") as fh:
                    r = self.s.put(f"{dep['links']['bucket']}/{name}", data=fh,
                                   headers={"Content-Type": "application/octet-stream"}, timeout=(60, None))
                if r.status_code >= 500:
                    raise RuntimeError(f"HTTP {r.status_code}")
                r.raise_for_status()
                remote = r.json().get("checksum", "")
                if remote.replace("md5:", "") != local:
                    raise RuntimeError("checksum MISMATCH")
                mins = (time.time() - t0) / 60
                size = os.path.getsize(path)
                if verbose:
                    print(f"  uploaded {name} ({size / 1e9:.2f} GB, {mins:.1f} min, "
                          f"{size / 1e6 / max(mins * 60, 1e-9):.2f} MB/s) checksum OK", flush=True)
                return remote
            except Exception as e:      # noqa: BLE001 - every failure is retried, then re-raised
                last = e
                if verbose:
                    print(f"  {name}: attempt {attempt}/{attempts} failed ({str(e)[:120]})", flush=True)
                if attempt < attempts:
                    time.sleep(wait * attempt)
        raise RuntimeError(f"{name} did not upload after {attempts} attempts: {last}")

    def open_drafts(self):
        r = self.s.get(f"{self.base}/deposit/depositions", params={"size": 50})
        r.raise_for_status()
        return [d for d in r.json() if not d.get("submitted")]

    def set_metadata(self, dep_id, metadata):
        r = self.s.put(f"{self.base}/deposit/depositions/{dep_id}", json={"metadata": metadata})
        if r.status_code >= 400:
            raise RuntimeError(f"metadata rejected ({r.status_code}): {r.text[:500]}")
        return r.json()


def mirror(src_dir, dst_dir, verbose=True):
    """Copy the export folder (archives, manifest, sidecars, README, code snapshot)
    to a second location, e.g. Google Drive, so that a runtime reset does not
    force a rebuild. Each copy is checked against the MD5 of its source; a file
    already present with the same size and MD5 is skipped."""
    import shutil
    os.makedirs(dst_dir, exist_ok=True)
    for p in sorted(glob.glob(os.path.join(src_dir, "*"))):
        if not os.path.isfile(p) or p.endswith(".part"):
            continue
        q = os.path.join(dst_dir, os.path.basename(p))
        want = md5_file(p)
        if os.path.exists(q) and os.path.getsize(q) == os.path.getsize(p) and md5_file(q) == want:
            if verbose:
                print(f"  {os.path.basename(p)}: already on the mirror - skipped", flush=True)
            continue
        t0 = time.time()
        shutil.copyfile(p, q + ".part")
        os.replace(q + ".part", q)
        if md5_file(q) != want:
            raise RuntimeError(f"copy of {os.path.basename(p)} to {dst_dir} is corrupt")
        if verbose:
            print(f"  {os.path.basename(p)}: copied ({os.path.getsize(q) / 1e9:.2f} GB, "
                  f"{(time.time() - t0) / 60:.1f} min)", flush=True)


def _upload_files(out_dir):
    files = sorted(p for p in glob.glob(os.path.join(out_dir, "*"))
                   if os.path.isfile(p) and not p.endswith((".members.json", ".part", ".state.json")))
    if len(files) > 100:
        raise RuntimeError(f"{len(files)} files: Zenodo allows at most 100 per record")
    return files


def _doi(dep):
    return ((dep.get("metadata") or {}).get("prereserve_doi") or {}).get("doi")


def upload_all(out_dir, token, metadata, sandbox=False, dep_id=None, verbose=True,
               state_path=None, n_parallel=1, new_draft=False):
    """Upload every file in out_dir (except sidecars) to a Zenodo DRAFT and set its
    metadata. Never publishes.

    Which draft: `dep_id` if given; otherwise the draft recorded in `state_path`
    (written as soon as a draft is created, so a disconnect never loses it);
    otherwise an open draft with the same title. If none of these exists but the
    account has other open drafts, it stops and lists them rather than create a
    second draft with a second DOI, unless `new_draft=True`.

    Re-running skips files already in the draft with a matching checksum. Each
    file is retried on failure (Zenodo.upload); with n_parallel > 1 several files
    upload at once, each over its own connection."""
    zc = Zenodo(token, sandbox=sandbox)
    md = dict(metadata); md["prereserve_doi"] = True
    dep = None
    if dep_id:
        dep = zc.get(dep_id)
    elif state_path and os.path.exists(state_path):
        st = json.load(open(state_path))
        if st.get("sandbox") == bool(sandbox):
            d = zc.get(st["id"])
            if not d.get("submitted"):
                dep = d
                if verbose:
                    print(f"reusing draft {dep['id']} recorded in {state_path}", flush=True)
    if dep is None and not new_draft:
        drafts = zc.open_drafts()
        same = [d for d in drafts if (d.get("title") or (d.get("metadata") or {}).get("title")) == md.get("title")]
        if same:
            dep = zc.get(same[0]["id"])
            if verbose:
                print(f"reusing open draft {dep['id']} with the same title", flush=True)
        elif drafts:
            lines = "\n".join(f"  {d['id']}  {d.get('title') or '(no title yet)'}  "
                              f"{len(d.get('files', []))} files  created {d.get('created', '')[:16]}" for d in drafts)
            raise RuntimeError("This account already has open drafts:\n" + lines +
                               "\nSet DEPOSITION_ID to the one to continue, or NEW_DRAFT = True to start another.")
    if dep is None:
        dep = zc.create_draft()
    if dep.get("submitted"):
        raise RuntimeError(f"deposition {dep['id']} is already published; its files cannot change")
    dep = zc.set_metadata(dep["id"], md)            # title + reserved DOI visible from the start
    doi = _doi(dep)
    url = f"{zc.web}/deposit/{dep['id']}"
    if state_path:
        os.makedirs(os.path.dirname(os.path.abspath(state_path)), exist_ok=True)
        with open(state_path, "w") as fh:
            json.dump({"id": dep["id"], "doi": doi, "url": url, "sandbox": bool(sandbox),
                       "saved_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}, fh, indent=1)
    if verbose:
        print(f"draft {dep['id']}  reserved DOI {doi}\n  {url}" + (f"\n  saved to {state_path}" if state_path else ""),
              flush=True)

    have = zc.uploaded(dep)
    todo, md5s = [], {}
    for p in _upload_files(out_dir):
        name = os.path.basename(p)
        local = md5s[name] = md5_file(p)
        if have.get(name) and have[name].replace("md5:", "") == local:
            if verbose:
                print(f"  {name}: already in the draft - skipped", flush=True)
        else:
            todo.append((p, local))
    gb = sum(os.path.getsize(p) for p, _ in todo) / 1e9
    if verbose:
        print(f"{len(todo)} files to upload ({gb:.2f} GB), {max(1, n_parallel)} at a time", flush=True)
    t0, failed = time.time(), []
    if n_parallel <= 1:
        for p, local in todo:
            try:
                zc.upload(dep, p, verbose=verbose, local_md5=local)
            except Exception as e:      # noqa: BLE001
                failed.append(os.path.basename(p)); print(f"  FAILED {os.path.basename(p)}: {e}", flush=True)
    else:
        from concurrent.futures import ThreadPoolExecutor, as_completed
        with ThreadPoolExecutor(max_workers=n_parallel) as ex:
            futs = {ex.submit(Zenodo(token, sandbox=sandbox).upload, dep, p, verbose, 3, 20, local): p
                    for p, local in todo}
            for f in as_completed(futs):
                try:
                    f.result()
                except Exception as e:  # noqa: BLE001
                    failed.append(os.path.basename(futs[f])); print(f"  FAILED {os.path.basename(futs[f])}: {e}", flush=True)
    mins = (time.time() - t0) / 60
    if failed:
        raise RuntimeError(f"{len(failed)} file(s) failed: {failed}. Re-run: finished files are skipped "
                           f"and the same draft ({dep['id']}) is reused.")
    final = zc.uploaded(zc.get(dep["id"]))
    missing = [n for n, m in md5s.items() if (final.get(n) or "").replace("md5:", "") != m]
    if missing:
        raise RuntimeError(f"not in the draft with a matching checksum: {missing}; re-run")
    if verbose:
        print(f"\nall {len(final)} files in the draft, checksums match "
              f"({gb * 1000 / max(mins * 60, 1e-9):.2f} MB/s over {mins:.1f} min for this run)")
        print(f"DRAFT ready (not published): {url}\nReserved DOI: {doi}")
    return {"id": dep["id"], "url": url, "doi": doi}

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/fetch_features.py
"""Download the features and held-out logits from Zenodo, verify every checksum,
and set up a project folder in which notebooks 04-08 run unchanged.

    python code/fetch_features.py --record 1234567 --root ~/cka_study
    export V3_DRIVE=~/cka_study          # the notebooks' config reads this
    python code/verify_v3.py --root .    # still works from the repository alone

Resulting layout (the one the code expects under <root>/TMLR_v3):

    code/                       copied from this repository (hash-checked by the registration)
    prereg.json                 the registration
    manifests/                  row order, labels and groups of every dataset
    features/<dataset>/<encoder>.npy, .json
    results/pools/<dataset>.json, <dataset>_logits.npz
"""

from __future__ import annotations

import argparse
import glob
import hashlib
import json
import os
import shutil
import sys
import time
import zipfile

CHUNK = 1 << 22
REPO = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for b in iter(lambda: fh.read(CHUNK), b""):
            h.update(b)
    return h.hexdigest()


def record_files(record, sandbox=False, session=None):
    import requests
    s = session or requests.Session()
    base = "https://sandbox.zenodo.org" if sandbox else "https://zenodo.org"
    r = s.get(f"{base}/api/records/{record}", timeout=60)
    r.raise_for_status()
    out = []
    for f in r.json().get("files", []):
        name = f.get("key") or f.get("filename")
        link = (f.get("links") or {}).get("self") or (f.get("links") or {}).get("download")
        out.append({"name": name, "url": link, "size": f.get("size"), "checksum": f.get("checksum")})
    return out


def download(url, dst, session=None, retries=5):
    """Resumable download (HTTP Range); keeps <dst>.part until complete."""
    import requests
    s = session or requests.Session()
    part = dst + ".part"
    for attempt in range(retries):
        have = os.path.getsize(part) if os.path.exists(part) else 0
        try:
            with s.get(url, stream=True, timeout=120,
                       headers={"Range": f"bytes={have}-"} if have else {}) as r:
                if r.status_code == 416:
                    break
                r.raise_for_status()
                mode = "ab" if have and r.status_code == 206 else "wb"
                with open(part, mode) as fh:
                    for b in r.iter_content(CHUNK):
                        fh.write(b)
            break
        except Exception as e:
            print(f"  retry {attempt + 1}/{retries} for {os.path.basename(dst)}: {e}")
            time.sleep(10)
    os.replace(part, dst)
    return dst


def setup_project(root):
    """Copy what the notebooks need from this repository into <root>/TMLR_v3."""
    proj = os.path.join(root, "TMLR_v3")
    for d in ("code", "manifests", os.path.join("results", "pools"), "features", "logs"):
        os.makedirs(os.path.join(proj, d), exist_ok=True)
    for p in glob.glob(os.path.join(REPO, "code", "*.py")):
        shutil.copy2(p, os.path.join(proj, "code"))
    for p in glob.glob(os.path.join(REPO, "manifests", "*")):
        shutil.copy2(p, os.path.join(proj, "manifests"))
    for p in glob.glob(os.path.join(REPO, "results", "pools", "*.json")):
        shutil.copy2(p, os.path.join(proj, "results", "pools"))
    for p in ("battery.json", "paper_tables.json"):
        src = os.path.join(REPO, "results", p)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(proj, "results"))
    shutil.copy2(os.path.join(REPO, "registration", "prereg.json"), os.path.join(proj, "prereg.json"))
    return proj


def unpack(zip_path, proj, members_sha):
    """Extract an archive, re-hashing every member against the manifest."""
    with zipfile.ZipFile(zip_path) as z:
        for arc in z.namelist():
            if arc.startswith("pools/"):
                dst = os.path.join(proj, "results", arc)
            else:
                dst = os.path.join(proj, "features", arc)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            with z.open(arc) as src, open(dst + ".part", "wb") as out:
                shutil.copyfileobj(src, out, CHUNK)
            os.replace(dst + ".part", dst)
            want = members_sha.get(arc)
            if want and sha256_file(dst) != want:
                raise RuntimeError(f"checksum mismatch after unpacking {arc}")


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--record", required=True, help="Zenodo record id")
    ap.add_argument("--root", required=True, help="folder to create the project in")
    ap.add_argument("--only", nargs="*", default=None,
                    help="dataset names to fetch (default: all); logits are always fetched")
    ap.add_argument("--sandbox", action="store_true")
    ap.add_argument("--keep-archives", action="store_true")
    a = ap.parse_args(argv)
    root = os.path.abspath(os.path.expanduser(a.root))
    dl = os.path.join(root, "zenodo_download")
    os.makedirs(dl, exist_ok=True)
    proj = setup_project(root)
    files = {f["name"]: f for f in record_files(a.record, a.sandbox)}
    if "zenodo_manifest.json" not in files:
        sys.exit("record has no zenodo_manifest.json")
    man_path = download(files["zenodo_manifest.json"]["url"], os.path.join(dl, "zenodo_manifest.json"))
    man = json.load(open(man_path))
    wanted = [n for n in man["archives"]
              if n == "heldout_logits.zip" or a.only is None
              or n[len("features_"):-len(".zip")] in set(a.only)]
    for name in wanted:
        dst = os.path.join(dl, name)
        rec = man["archives"][name]
        if not (os.path.exists(dst) and sha256_file(dst) == rec["archive_sha256"]):
            print(f"downloading {name} ({rec['archive_bytes'] / 1e9:.2f} GB) ...", flush=True)
            download(files[name]["url"], dst)
        if sha256_file(dst) != rec["archive_sha256"]:
            sys.exit(f"{name}: archive checksum mismatch")
        unpack(dst, proj, {k: v["sha256"] for k, v in rec["members"].items()})
        print(f"  {name}: verified and unpacked ({len(rec['members'])} files)", flush=True)
        if not a.keep_archives:
            os.remove(dst)
    print(f"\nproject ready: {proj}\nnext: export V3_DRIVE={root}  (then run notebooks 04-08 or code/ directly)")


if __name__ == "__main__":
    main()

## 1. Settings

In [ ]:
import config as CFG, export_zenodo as EZ
for m in (CFG, EZ):
    importlib.reload(m)

GITHUB_URL  = "https://github.com/munna237/cka-frozen-encoder-selection"
PAPER_TITLE = ("Evaluating Centered Kernel Alignment for Frozen-Encoder Selection in Logit-Level Fusion: "
               "A Pre-Specified Multi-Domain Study")
LICENSE = "cc-by-nc-4.0"
GATED = sorted(k for k, v in CFG.PATHOLOGY_ENCODERS.items() if v.get("gated"))
EXCLUDE_ENCODERS = GATED          # features AND held-out logits of these are left out
EXCLUDE_DATASETS = ()
INCLUDE_CODE_SNAPSHOT = True      # a zip of the GitHub repository's main branch
OUT_DIR = "/content/zenodo_export"                       # built here (fast local disk)
KEEP_ON_DRIVE = True                                     # also keep a verified copy on Drive
DRIVE_EXPORT = os.path.join(PROJECT, "zenodo_export")    # survives a runtime reset
print("excluded (gated):", EXCLUDE_ENCODERS)

feats, logits = EZ.inventory(EXCLUDE_DATASETS, EXCLUDE_ENCODERS)
S = EZ.summary(feats, logits)
print(f"{S['datasets']} datasets, {S['feature_files']} feature files, {S['logits_files']} logits files, "
      f"{S['total_gb']} GB to package")
print(f"free local disk: {shutil.disk_usage('/content').free / 1e9:.0f} GB")
if KEEP_ON_DRIVE:
    print(f"free Drive space: {shutil.disk_usage('/content/drive').free / 1e9:.0f} GB "
          f"(the copy needs about {S['total_gb']} GB; set KEEP_ON_DRIVE = False to skip it)")

## 2. Find the Zenodo token

In [ ]:
from google.colab import userdata
NAMES = ["ZENODO_TOKEN", "ZENODO_API_KEY", "ZENODO_KEY", "ZENODO_ACCESS_TOKEN", "ZENODO", "zenodo_token",
         "zenodo", "ZENODO_SANDBOX_TOKEN"]
TOKEN, TOKEN_NAME = None, None
for n in NAMES:
    try:
        v = userdata.get(n)
    except Exception:
        continue
    if v:
        TOKEN, TOKEN_NAME = v.strip(), n
        break
assert TOKEN, ("No Zenodo token found in Colab Secrets under: " + ", ".join(NAMES) +
               ". Check the secret's name and that 'Notebook access' is switched on.")
SITE = EZ.detect_site(TOKEN)
assert SITE, ("The token was rejected by both zenodo.org and sandbox.zenodo.org. Create a new one with the "
              "scopes deposit:write and deposit:actions.")
SANDBOX = SITE == "sandbox"
print(f"token from secret '{TOKEN_NAME}' works on: {'sandbox.zenodo.org (test site)' if SANDBOX else 'zenodo.org'}")

## 3. Build, verify and copy the archives to Drive (resumable: finished archives are skipped)

In [ ]:
import zipfile, numpy as np
LOCAL_OK = os.path.exists(os.path.join(OUT_DIR, "zenodo_manifest.json"))
DRIVE_OK = KEEP_ON_DRIVE and os.path.exists(os.path.join(DRIVE_EXPORT, "zenodo_manifest.json"))
if DRIVE_OK and not LOCAL_OK:
    # fresh runtime after a reset: use the copy on Drive instead of rebuilding.
    # Each archive is checked against the size and MD5 recorded when it was built.
    UPLOAD_DIR = DRIVE_EXPORT
    man = json.load(open(os.path.join(UPLOAD_DIR, "zenodo_manifest.json")))
    for name, rec in {**man["archives"], **man.get("other_files", {})}.items():
        p = os.path.join(UPLOAD_DIR, name)
        assert os.path.exists(p) and os.path.getsize(p) == rec.get("archive_bytes", rec.get("bytes")), f"{name} missing or wrong size"
        assert EZ.md5_file(p) == rec.get("archive_md5", rec.get("md5")), f"{name}: MD5 differs from the build record"
        print(f"  {name}: OK on Drive")
    assert os.path.exists(os.path.join(UPLOAD_DIR, "README.md"))
    CODE_ZIP = next(iter(man.get("other_files", {})), None)
    print("using the verified copy on Drive:", UPLOAD_DIR)
else:
    man = EZ.build(OUT_DIR, feats, logits, include_logits=True, exclude_logits_of=EXCLUDE_ENCODERS)
    CODE_ZIP = None
    if INCLUDE_CODE_SNAPSHOT:
        import urllib.request
        CODE_ZIP = "cka-frozen-encoder-selection-code.zip"
        tmp = os.path.join("/content", CODE_ZIP)
        urllib.request.urlretrieve(GITHUB_URL.replace("github.com", "codeload.github.com") + "/zip/refs/heads/main", tmp)
        EZ.add_file(OUT_DIR, tmp, CODE_ZIP)
        print("code snapshot:", round(os.path.getsize(tmp) / 1e6, 1), "MB")
    EZ.write_readme(OUT_DIR, "Frozen-encoder features and held-out logits", GITHUB_URL, PAPER_TITLE, S,
                    excluded=EXCLUDE_ENCODERS, code_snapshot=CODE_ZIP)
    problems = EZ.verify(OUT_DIR)
    assert not problems, problems
    print("all archives verified")
    UPLOAD_DIR = OUT_DIR
    if KEEP_ON_DRIVE:
        print("\ncopying to Drive (each copy checked by MD5) ...")
        EZ.mirror(OUT_DIR, DRIVE_EXPORT)
        print("copy on Drive:", DRIVE_EXPORT)

# the gated encoders must be absent from every archive that will be uploaded
for name in man["archives"]:
    with zipfile.ZipFile(os.path.join(UPLOAD_DIR, name)) as z:
        bad = [m for m in z.namelist() if any(f"/{e}." in m for e in EXCLUDE_ENCODERS)]
        assert not bad, (name, bad[:3])
        if name == "heldout_logits.zip":
            for m in z.namelist():
                with z.open(m) as fh:
                    keys = np.load(fh).files
                assert not [k for k in keys if k[2:] in EXCLUDE_ENCODERS], (m, keys)
print("no gated encoder in any archive")
print("upload from:", UPLOAD_DIR)
print(sorted(f for f in os.listdir(UPLOAD_DIR) if not f.endswith(".members.json") and not f.startswith("_")))

## 4. Upload to a Zenodo draft
Creates the draft (or reuses the one saved in `TMLR_v3/zenodo_draft.json`), **reserves the DOI**, fills in the
metadata, then uploads every file, checking each checksum and retrying failures. It never publishes.

* **Interrupted?** Just re-run this cell: the same draft is reused and finished files are skipped.
* **The cell stops with "already has open drafts"?** That happens when no saved draft is found but the account has
  other drafts, for example one started by an earlier version of this notebook. Set `DEPOSITION_ID` to the one to
  continue (the listing shows its files), or `NEW_DRAFT = True` to start a new draft with a new DOI.
* **Slow?** Raise `N_PARALLEL` to 3 to send several files at once. That helps only if a single connection is the
  limit; the printed MB/s shows whether it did.

Check the creators' names, affiliations and ORCIDs below; they are taken from the paper.

In [ ]:
DEPOSITION_ID = None     # set to continue a specific draft (normally found automatically)
NEW_DRAFT = False        # True only to deliberately start a second draft with a new DOI
N_PARALLEL = 1           # files uploaded at once; try 3 if uploads are slow
STATE_PATH = os.path.join(PROJECT, f"zenodo_draft{'_sandbox' if SANDBOX else ''}.json")

metadata = {
    "title": "Frozen-encoder features and held-out logits for: " + PAPER_TITLE,
    "upload_type": "dataset",
    "description": open(os.path.join(UPLOAD_DIR, "README.md")).read().replace("\n", "<br>"),
    "creators": [
        {"name": "Munna, Mohammad Mehedi Hasan", "affiliation": "North South University", "orcid": "0009-0000-0860-9337"},
        {"name": "Kabir, Acramul Haque", "affiliation": "North South University", "orcid": "0000-0002-4899-5786"},
        {"name": "Tulona, Reefka Fabliha", "affiliation": "East West University", "orcid": "0009-0002-3557-0938"},
        {"name": "Jawad, Md Asef", "affiliation": "North South University", "orcid": "0009-0008-5551-5163"},
        {"name": "Antu, Nahid Hossain", "affiliation": "North South University", "orcid": "0009-0007-5924-6901"},
        {"name": "Ul Mulk, Md Mansur", "affiliation": "North South University", "orcid": "0009-0000-0743-3309"},
        {"name": "Chowdhury, Arvid", "affiliation": "North South University"},
    ],
    "license": LICENSE,
    "access_right": "open",
    "version": "1.0",
    "keywords": ["centered kernel alignment", "encoder selection", "model fusion", "representational similarity",
                 "computational pathology", "frozen features", "pre-registration"],
    "related_identifiers": [{"identifier": GITHUB_URL, "relation": "isSupplementTo", "resource_type": "software"}],
}
res = EZ.upload_all(UPLOAD_DIR, TOKEN, metadata, sandbox=SANDBOX, dep_id=DEPOSITION_ID,
                    state_path=STATE_PATH, n_parallel=N_PARALLEL, new_draft=NEW_DRAFT)
print("\nNext: open the draft, check it, put the DOI in the paper and the repository, then press Publish.")
print("Record id (for fetch_features.py):", res["id"])